In [1]:
import torch.nn as nn
import cv2



In [2]:
# Khoa_LHR_image.zip
!gdown --id 1bsWkNmmYvBrgE1c58SGJFcCjQv3SUyH3

/media/tan/F/AIO2024_hw/env/lib/python3.10/site-packages/gdown/__main__.py:140: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From (original): https://drive.google.com/uc?id=1bsWkNmmYvBrgE1c58SGJFcCjQv3SUyH3
From (redirected): https://drive.google.com/uc?id=1bsWkNmmYvBrgE1c58SGJFcCjQv3SUyH3&confirm=t&uuid=22ae84cc-3a03-4fe9-840e-14e1279fc1d1
To: /media/tan/F/AIO2024_hw/AIO2024_hw/Module7/week1/Khoa_LHR_image.zip
100%|██████████████████████████████████████| 89.0M/89.0M [00:03<00:00, 28.9MB/s]


In [2]:
import os
from PIL import Image

# Define the path to the validation and test pictures folder
origin_folder = 'Khoa_LHR_image'
origin_train_folder = os.path.join(origin_folder, 'train')
origin_val_folder = os.path.join(origin_folder, 'val')

# Define the path to the new folder to save the resized pictures
resized_folder = 'Resized_folder'
resized_train_folder = os.path.join(resized_folder, 'train')
resized_val_folder = os.path.join(resized_folder, 'val')

# Create the new folder if it doesn't exist
if not os.path.exists(resized_folder):
    os.makedirs(resized_folder)

if not os.path.exists(resized_train_folder):
    os.makedirs(resized_train_folder)

if not os.path.exists(resized_val_folder):
    os.makedirs(resized_val_folder)


def resize_images(origin_folder, resized_folder, size=(64, 64)):
    # Loop through the validation and test pictures folder
    for filename in os.listdir(origin_folder):
        # Load the image
        image_path = os.path.join(origin_folder, filename)
        image = Image.open(image_path)

        # Resize the image to the specified size
        resized_image = image.resize(size)

        # Convert the image to RGB if it's not already
        if resized_image.mode != 'RGB':
            resized_image = resized_image.convert('RGB')

        # Save the resized image to the new folder
        resized_image.save(os.path.join(resized_folder, filename))


resize_images(origin_train_folder, resized_train_folder)
resize_images(origin_val_folder, resized_val_folder)

In [ ]:
train_data_path =  resized_train_folder
val_data_path = resized_val_folder

batch_size = 16

In [14]:
import torch

class FirstFeature(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(FirstFeature, self).__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 1, 1, 0, bias = False),
            nn.LeakyReLU()
        )

    def forward(self, x):
        return self.conv(x)
    
class ConvBlock(nn.Module):
    def __init__(self,in_channels, out_channels):
        super(ConvBlock, self).__init__()
        self.conv = nn.Sequential(nn.Conv2d(in_channels, out_channels,3,1,1,bias= False),
                                nn.BatchNorm2d(out_channels),
                                nn.LeakyReLU(inplace = True),
                                nn.Conv2d(out_channels, out_channels, 3,1,1, bias = False),
                                nn.BatchNorm2d(out_channels),
                                nn.LeakyReLU(inplace = True)
                                )
        
    def forward(self, x):
        return self.conv(x)
    
class Encoder(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.MaxPool2d(2),
            ConvBlock(in_channels, out_channels)
        )

    def forward(self, x):
        x = self.encoder(x)
        return x
    
class Decoder(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(Decoder, self).__init__()
        self.conv = nn.Sequential(
            nn.UpsamplingBilinear2d(scale_factor=2),
            nn.Conv2d(in_channels, out_channels,1,1,0, bias = False),
            nn.BatchNorm2d(out_channels),
            nn.LeakyReLU()
            )
        self.conv_block = ConvBlock(in_channels, out_channels)

    def forward(self,x, skip):
        x = self.conv(x)
        x = torch.concat([x, skip], dim = 1)
        x = self.conv_block(x)
        return x

class FinalOutput(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(FinalOutput, self).__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=1, padding=0, bias=False),
            nn.Tanh()
        )

    def forward(self, x):
        return self.conv(x)


class Unet(nn.Module):
    def __init__(self, n_channels=3, n_classes=3, features=[64, 128, 256, 512]):
        super(Unet, self).__init__()
        self.n_channels = n_channels
        self.n_classes = n_classes

        self.in_conv1 = FirstFeature(n_channels, 64)
        self.in_conv2 = ConvBlock(64, 64)

        self.enc_1 = Encoder(64, 128)
        self.enc_2 = Encoder(128, 256)
        self.enc_3 = Encoder(256, 512)
        self.enc_4 = Encoder(512, 1024)

        self.dec_1 = Decoder(1024, 512)
        self.dec_2 = Decoder(512, 256)
        self.dec_3 = Decoder(256, 128)
        self.dec_4 = Decoder(128, 64)

        self.out_conv = FinalOutput(64, n_classes)

    def forward(self, x):
        x = self.in_conv1(x)
        x1 = self.in_conv2(x)

        x2 = self.enc_1(x1)
        x3 = self.enc_2(x2)
        x4 = self.enc_3(x3)
        x5 = self.enc_4(x4)

        x = self.dec_1(x5, x4)
        x = self.dec_2(x, x3)
        x = self.dec_3(x, x2)
        x = self.dec_4(x, x1)

        x = self.out_conv(x)
        return x




In [15]:
from torchsummary import summary

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = Unet().to(device)

In [16]:
summary(model, (3,256,256))

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1         [-1, 64, 256, 256]             192
         LeakyReLU-2         [-1, 64, 256, 256]               0
      FirstFeature-3         [-1, 64, 256, 256]               0
            Conv2d-4         [-1, 64, 256, 256]          36,864
       BatchNorm2d-5         [-1, 64, 256, 256]             128
         LeakyReLU-6         [-1, 64, 256, 256]               0
            Conv2d-7         [-1, 64, 256, 256]          36,864
       BatchNorm2d-8         [-1, 64, 256, 256]             128
         LeakyReLU-9         [-1, 64, 256, 256]               0
        ConvBlock-10         [-1, 64, 256, 256]               0
        MaxPool2d-11         [-1, 64, 128, 128]               0
           Conv2d-12        [-1, 128, 128, 128]          73,728
      BatchNorm2d-13        [-1, 128, 128, 128]             256
        LeakyReLU-14        [-1, 128, 1